# 11 — Stacked Meta-Learner Ensemble

Trains a RidgeCV meta-learner on out-of-fold (OOF) predictions from four base models:

| Model | OOF RAE (ref) | Notes |
|---|---|---|
| LGBM base | ~0.575 | raw features, no augmentation |
| LGBM aug | 0.5582 | + null feature + upsampling |
| kNN (k=20) | 0.733 (scaffold CV underestimate) | Tanimoto-weighted |
| LGBM pipeline | ? | cleaned data + feature selection (nb 09) |

Final submission blends the meta-learner output with Chemprop (from nb 08/10) via inverse-RAE weights.

**Runtime**: ~25 min (all base models are fast; Chemprop is loaded from saved submissions).

In [ ]:
# ── 1. Setup ──────────────────────────────────────────────────────────────────
import sys, warnings, pickle
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
from sklearn.linear_model import RidgeCV
from sklearn.neighbors import NearestNeighbors

from pxr.data import load_train, load_test, load_counter
from pxr.chem import to_inchikey, bemis_murcko, morgan_fp_batch
from pxr.featurize import combined, impute
from pxr.eval import scaffold_kfold_indices, compute_metrics, rae as rae_fn
from pxr.paths import DATA_PROCESSED, SUBMISSIONS

plt.rcParams.update({"figure.dpi": 120})
SEED = 42
print("Setup complete.")

In [ ]:
# ── 2. Load training data ─────────────────────────────────────────────────────
train_raw = load_train()
counter   = load_counter()
te        = load_test()

# Use raw training set (all 4,139) as the OOF base — consistent across all models
smiles_tr = train_raw['smiles'].tolist()
y_tr      = train_raw['pec50'].values

scaffolds = train_raw['smiles'].map(bemis_murcko).tolist()
splits    = scaffold_kfold_indices(scaffolds, n_splits=5, seed=SEED)

# Null-assay feature (same recipe as nb 06)
tr_ik  = train_raw.assign(inchikey=train_raw['smiles'].map(to_inchikey))
ct_ik  = counter.assign(inchikey=counter['smiles'].map(to_inchikey))
joined = tr_ik.merge(ct_ik[['inchikey','pec50']].rename(columns={'pec50':'pec50_null'}),
                     on='inchikey', how='left')
has_null = joined['pec50_null'].notna()

print(f"Training: {len(train_raw):,}  |  null coverage: {has_null.mean():.1%}")

In [ ]:
# ── 3. Compute base features ──────────────────────────────────────────────────
print("Computing combined features (train) ...")
X_base = impute(combined(smiles_tr))

# Null feature: impute missing with fold-level null predictor
pec50_null = joined['pec50_null'].values.copy()

# Morgan FP for kNN
X_fp_tr = morgan_fp_batch(smiles_tr).astype(bool)
X_fp_te = morgan_fp_batch(te['smiles'].tolist()).astype(bool)

print(f"Base features: {X_base.shape}")

# Test features
print("Computing combined features (test) ...")
X_te_base = impute(combined(te['smiles'].tolist()))

In [ ]:
# ── 4. LGBM hyperparams ───────────────────────────────────────────────────────
lgbm_params = dict(
    n_estimators=1000, num_leaves=64, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=0.1,
    min_child_samples=10, n_jobs=4, verbose=-1,
)

In [ ]:
# ── 5. OOF: LGBM base ─────────────────────────────────────────────────────────
oof_lgbm_base = np.full(len(y_tr), np.nan)
for tr_idx, va_idx in splits:
    m = lgb.LGBMRegressor(**lgbm_params)
    m.fit(X_base[tr_idx], y_tr[tr_idx])
    oof_lgbm_base[va_idx] = m.predict(X_base[va_idx])
print(f"LGBM base OOF RAE: {rae_fn(y_tr, oof_lgbm_base):.4f}")

In [ ]:
# ── 6. OOF: LGBM aug (+ null feature + upsampling) ───────────────────────────
from pxr.preprocess import upsample_by_category

oof_lgbm_aug = np.full(len(y_tr), np.nan)
for tr_idx, va_idx in splits:
    # Impute null pEC50 for this fold's training compounds
    null_feat = pec50_null.copy()
    tr_missing = tr_idx[np.isnan(pec50_null[tr_idx])]
    if len(tr_missing) > 0:
        null_imp = lgb.LGBMRegressor(n_estimators=300, verbose=-1)
        has_val  = tr_idx[~np.isnan(pec50_null[tr_idx])]
        null_imp.fit(X_base[has_val], pec50_null[has_val])
        null_feat[tr_missing] = null_imp.predict(X_base[tr_missing])

    X_aug = np.hstack([X_base, null_feat.reshape(-1, 1)])
    X_tr_up, y_tr_up = upsample_by_category(X_aug[tr_idx], y_tr[tr_idx])
    m = lgb.LGBMRegressor(**lgbm_params)
    m.fit(X_tr_up, y_tr_up)
    oof_lgbm_aug[va_idx] = m.predict(X_aug[va_idx])
print(f"LGBM aug  OOF RAE: {rae_fn(y_tr, oof_lgbm_aug):.4f}")

In [ ]:
# ── 7. OOF: kNN (k=20, Tanimoto) ─────────────────────────────────────────────
oof_knn = np.full(len(y_tr), np.nan)
for tr_idx, va_idx in splits:
    nn = NearestNeighbors(n_neighbors=20, metric='jaccard', algorithm='brute', n_jobs=4)
    nn.fit(X_fp_tr[tr_idx])
    dists, idxs = nn.kneighbors(X_fp_tr[va_idx])
    sims = 1.0 - dists
    w    = sims / (sims.sum(axis=1, keepdims=True) + 1e-9)
    oof_knn[va_idx] = (w * y_tr[tr_idx][idxs]).sum(axis=1)
print(f"kNN       OOF RAE: {rae_fn(y_tr, oof_knn):.4f}")

In [ ]:
# ── 8. OOF: LGBM pipeline (nb 09 cleaned data) ────────────────────────────────
try:
    X_sel    = np.load(DATA_PROCESSED / 'X_train_sel.npy')
    y_clean  = np.load(DATA_PROCESSED / 'y_train_clean.npy')
    with open(DATA_PROCESSED / 'feature_selector.pkl', 'rb') as f:
        selector = pickle.load(f)
    X_te_sel = np.load(DATA_PROCESSED / 'X_test_sel.npy')

    # Scaffold splits on the cleaned set
    clean_df     = pd.read_parquet(DATA_PROCESSED / 'train_clean.parquet')
    clean_scaf   = clean_df['smiles'].map(bemis_murcko).tolist()
    splits_clean = scaffold_kfold_indices(clean_scaf, n_splits=5, seed=SEED)

    oof_lgbm_pipe = np.full(len(y_clean), np.nan)
    for tr_idx, va_idx in splits_clean:
        X_tr_up, y_tr_up = upsample_by_category(X_sel[tr_idx], y_clean[tr_idx])
        m = lgb.LGBMRegressor(**lgbm_params)
        m.fit(X_tr_up, y_tr_up)
        oof_lgbm_pipe[va_idx] = m.predict(X_sel[va_idx])

    pipeline_available = True
    print(f"LGBM pipe OOF RAE: {rae_fn(y_clean, oof_lgbm_pipe):.4f}")
except FileNotFoundError:
    pipeline_available = False
    print("LGBM pipeline data not found (run nb 09 first) — using 3-model stack")

In [ ]:
# ── 9. Stack OOF matrix + train RidgeCV meta-learner ─────────────────────────
# Use raw-set models (consistent 4,139 rows)
oof_stack = np.column_stack([oof_lgbm_base, oof_lgbm_aug, oof_knn])
y_meta    = y_tr

alphas = np.logspace(-3, 3, 100)
meta   = RidgeCV(alphas=alphas, cv=5)
meta.fit(oof_stack, y_meta)

oof_meta = meta.predict(oof_stack)
meta_rae = rae_fn(y_meta, oof_meta)

print(f"Meta-learner alpha: {meta.alpha_:.4f}")
print(f"Meta-learner weights: {dict(zip(['lgbm_base','lgbm_aug','knn'], meta.coef_.round(3)))}")
print(f"Meta OOF RAE (training set, in-sample): {meta_rae:.4f}")
print()
# True estimate: nested scaffold CV
oof_nested = np.full(len(y_tr), np.nan)
for tr_idx, va_idx in splits:
    m = RidgeCV(alphas=alphas, cv=3)
    m.fit(oof_stack[tr_idx], y_meta[tr_idx])
    oof_nested[va_idx] = m.predict(oof_stack[va_idx])
print(f"Nested scaffold CV meta OOF RAE: {rae_fn(y_tr, oof_nested):.4f}")

In [ ]:
# ── 10. Train final base models on all training data ──────────────────────────
# LGBM base
lgbm_base_final = lgb.LGBMRegressor(**lgbm_params)
lgbm_base_final.fit(X_base, y_tr)
te_lgbm_base = lgbm_base_final.predict(X_te_base)

# LGBM aug (with null feature imputed on full training set)
null_full = pec50_null.copy()
missing   = np.isnan(null_full)
if missing.any():
    null_imp_f = lgb.LGBMRegressor(n_estimators=300, verbose=-1)
    null_imp_f.fit(X_base[~missing], null_full[~missing])
    null_full[missing] = null_imp_f.predict(X_base[missing])
X_aug_full = np.hstack([X_base, null_full.reshape(-1, 1)])
X_te_aug   = np.hstack([X_te_base, null_imp_f.predict(X_te_base).reshape(-1, 1)])
X_tr_up_f, y_tr_up_f = upsample_by_category(X_aug_full, y_tr)
lgbm_aug_final = lgb.LGBMRegressor(**lgbm_params)
lgbm_aug_final.fit(X_tr_up_f, y_tr_up_f)
te_lgbm_aug = lgbm_aug_final.predict(X_te_aug)

# kNN on full training
nn_full = NearestNeighbors(n_neighbors=20, metric='jaccard', algorithm='brute', n_jobs=4)
nn_full.fit(X_fp_tr)
dists_te, idxs_te = nn_full.kneighbors(X_fp_te)
sims_te = 1.0 - dists_te
w_te    = sims_te / (sims_te.sum(axis=1, keepdims=True) + 1e-9)
te_knn  = (w_te * y_tr[idxs_te]).sum(axis=1)

print(f"Test preds: lgbm_base {te_lgbm_base.mean():.3f}  lgbm_aug {te_lgbm_aug.mean():.3f}  knn {te_knn.mean():.3f}")

In [ ]:
# ── 11. Apply meta-learner to test set ────────────────────────────────────────
te_stack = np.column_stack([te_lgbm_base, te_lgbm_aug, te_knn])
meta_te  = meta.predict(te_stack)

# Blend meta-learner with Chemprop (from best available Chemprop submission)
for fname in ['10_expanded_multitask.csv', '08_chemprop_cv_blend.csv',
              '07b_chemprop_weighted.csv', '03_ensemble_lgbm_chemprop.csv']:
    path = SUBMISSIONS / fname
    if path.exists():
        cp_sub   = pd.read_csv(path)
        cp_preds = cp_sub.set_index('Molecule Name').loc[te['name'].values, 'pEC50'].values
        cp_src   = fname
        break

print(f"Chemprop source: {cp_src}")

# Inverse-RAE blend: meta-learner RAE vs Chemprop RAE (08)
meta_rae_est   = rae_fn(y_tr, oof_nested)   # nested CV estimate
chemprop_rae   = 0.5736                      # from nb 08
w_cp = (1/chemprop_rae) / (1/chemprop_rae + 1/meta_rae_est)

final_preds = w_cp * cp_preds + (1 - w_cp) * meta_te
final_preds = np.clip(final_preds, y_tr.min() - 0.5, y_tr.max() + 0.5)

print(f"Meta RAE (nested CV): {meta_rae_est:.4f}  |  Chemprop RAE: {chemprop_rae:.4f}")
print(f"Chemprop weight: {w_cp:.3f}  |  Meta weight: {1-w_cp:.3f}")

In [ ]:
# ── 12. Save submission ───────────────────────────────────────────────────────
sub = pd.DataFrame({'Molecule Name': te['name'].values,
                    'SMILES':        te['smiles'].values,
                    'pEC50':         final_preds})
assert len(sub) == 513 and sub['pEC50'].notna().all()
out = SUBMISSIONS / '11_stacked_ensemble.csv'
sub.to_csv(out, index=False)
print(f"Saved: {out}")
print(sub['pEC50'].describe().round(3))

# Also save OOF arrays for use by other notebooks
np.save(DATA_PROCESSED / 'oof_lgbm_base.npy',  oof_lgbm_base)
np.save(DATA_PROCESSED / 'oof_lgbm_aug.npy',   oof_lgbm_aug)
np.save(DATA_PROCESSED / 'oof_knn.npy',        oof_knn)
np.save(DATA_PROCESSED / 'oof_meta.npy',       oof_nested)
np.save(DATA_PROCESSED / 'te_lgbm_base.npy',   te_lgbm_base)
np.save(DATA_PROCESSED / 'te_lgbm_aug.npy',    te_lgbm_aug)
np.save(DATA_PROCESSED / 'te_knn.npy',         te_knn)
print("OOF and test arrays saved to data/processed/")